# L13b: Interpretable Allocation and Model-Predictive Rebalancing
Previously, we chose a portfolio by maximizing a utility function under a budget. How do we turn that one-time choice into a trading system that responds when prices and model estimates change?

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Build an adaptive allocation decision:__ Explain how new observations update the single index estimates and the market state. Connect those estimates to stock scores and constrained target shares.
> * __Close the loop with model-predictive rebalancing:__ Forecast the wealth of fixed holdings and compare it with the measured account. Follow a trigger through a new allocation to orders at the next open.
> * __Test which parts of the system help:__ Compare weighting rules, stock selection, and the market hedge on the same historical path. Read synthetic sell-off tests as evidence about behavior given a severe decline.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
>
> * New prices update the single index estimates and the market state. The resulting scores rank the stocks, while the budget and trading limits determine the target shares.
> * Updating the estimates and trading are separate decisions. The controller holds its shares until the horizon, a drawdown, or a forecast-band exit starts a new plan.
> * A test must hold the other choices fixed to measure what one component adds. Synthetic markets let us repeat that comparison under typical conditions and specified sell-offs.
>
> __For understanding__ (you will not need to reproduce or memorize these):
>
> * The weighted least-squares loss and the four utility functions.
> * The forecast simulation and the construction of the growth-rate covariance.
> * The INFORMS study's selection rules, optional market hedge, and construction of its sell-off sets.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L13b) lists this lecture's symbols with their meanings and units.

Let's get started!

___


## Examples

__We will work through these examples in lecture:__

> [▶ Trace an adaptive portfolio on real data](CHEME-5660-L13b-Example-MPC-RealData-Fall-2026.ipynb). In this example, we compute the market state and stock scores, run the constrained allocator inside a daily controller, and read its wealth and decision log. We then compare weighting rules, stock selection, and the hedge on the same historical path.

> [▶ Test the controller across synthetic markets](CHEME-5660-L13b-Example-MPC-SyntheticData-Fall-2026.ipynb). In this example, we draw typical markets and two kinds of sell-off from a fitted generator. We run the same controller on each market and compare its outcomes with SPY and an equal-weight portfolio.

___


## Review: Synthetic Markets with Jump Hidden Markov Models
Previously, we built synthetic markets from jump hidden Markov models (JumpHMM). The model bins the daily growth rate into states of equal probability and estimates the transition matrix by counting moves between states.

Each simulated day, the chain takes one step. The model then draws the day's growth rate from the Student-t distribution of the new state:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13b-JumpHMM-Schematic.svg" width="880" style="max-width:100%; height:auto;" alt="A row of hidden states numbered 1, 2, through k, through N minus 1 and N, ordered from the most negative to the most positive daily growth rate, with double-headed arrows between neighbors. States 1 and 2 are red and braced as the most negative states S minus, and states N minus 1 and N are blue and braced as the most positive states S plus. A bell curve above each state is its emission, and state k emits mu k plus sigma k times a Student-t variable. Under state k, a note reads ordinary day: step with the estimated transition matrix. A dashed arrow from state k, marked no episode under way, leads up to a box: jump episode, starts with probability epsilon, lasts a Poisson number of days with mean lambda. From the box, a red arrow runs to the most negative states, taken on each episode day with probability p neg, and a blue arrow runs to the most positive states, taken with probability 1 minus p neg."/>
    </center>
</div>

Jump episodes start at random and last several days. On each episode day, the state comes from the most negative or the most positive states. These runs of large moves add the volatility clustering that the chain alone misses.

To build a market of many stocks, we gave each synthetic stock a baseline, shared market and sector draws, and its own residual draw. The shared draws make the synthetic stocks move together like real ones.

__Today:__ Our goal is a trade management system that makes money when it can and, at a minimum, preserves cash when it cannot. We build that system and test it on one historical path and on many synthetic markets.

___


## Company Profile: AQR Capital Management
[AQR Capital Management](https://www.aqr.com/Our-Firm/About-Us) is a quantitative investment management firm. It was founded in New York City in 1998 and moved its headquarters to Greenwich, Connecticut, in 2004. The firm describes [its approach](https://www.aqr.com/What-We-Do/Our-Approach) as "systematic investing grounded in economic theory."

> __What does the firm do?__
>
> * __Betting against beta:__ In [Betting Against Beta](https://www.aqr.com/Insights/Research/Journal-Article/Betting-Against-Beta), published in 2014, AQR's Andrea Frazzini and Lasse H. Pedersen found that stocks with high beta tend to have low alpha. A portfolio that bought low-beta stocks with borrowed money and sold high-beta stocks short earned positive risk-adjusted returns.
> * __Trading costs:__ In [Trading Costs](https://www.aqr.com/Insights/Research/Working-Paper/Trading-Costs) (2018), Andrea Frazzini, Ronen Israel, and Tobias Moskowitz studied 1.7 trillion USD of executed trades in 21 developed stock markets over 19 years. They found actual trading costs to be "an order of magnitude smaller than previous studies suggest."

__Explore further:__

* __Jobs and internships:__ AQR's [internships page](https://www.aqr.com/Our-Firm/Greenwich-Internships) describes a 10-week summer program in Greenwich. In November 2026, the [careers site](https://careers.aqr.com/) listed 2027 summer analyst roles in research, engineering, and business development.
* __Video:__ Watch [About AQR](https://www.youtube.com/embed/946GJrhyZtM) and [Internships at AQR](https://www.youtube.com/embed/Oow2UktGKcY) on the firm's YouTube channel.

__Connection to today's lecture:__ When the market falls, our stock score favors stocks with positive expected growth and small $|\beta_i|$, the side that the betting-against-beta portfolio buys. Each new allocation also caps its trading cost, the quantity the Trading Costs paper measured on real trades.

___


## Building an Adaptive Trading System
A utility maximization tells us what to hold at one set of prices and model estimates. Once we buy those shares, prices keep moving and new observations change the estimates. Repeating the allocation requires rules for what to update and when to trade.

The system we build comes from J. D. Varner's INFORMS 2026 study, [Interpretable Allocation and Model-Predictive Rebalancing of Equity Portfolios](https://arxiv.org/abs/2611.03022). The [study repository](https://github.com/varnerlab/INFORMS-Annual-Meeting-2026-Varnerlab) contains its code and results.

The stocks and cash we hold form the __book__. Let $n_i$ be the shares held of stock $i$ in the basket $\mathcal N$, $S_i(t)$ its close price (units: USD/share), and $c(t)$ the cash balance (units: USD). The measured wealth of the book at the close of trading day $t$ is given by:

$$
W(t)=\sum_{i\in\mathcal N}n_iS_i(t)+c(t).
$$

The __controller__ is the system that decides, each trading day, what to update and whether to trade:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
<img class="course-diagram" src="figs/Fig-L13b-Portfolio-Feedback.svg" width="1050" style="max-width:100%; height:auto;" alt="At each close, the controller measures account wealth, updates estimates and scores, and checks the plan. Without a trigger it keeps the shares. A trigger starts an allocation, a new forecast, and orders for the next open. The next close returns to measurement."/>
</p>

At every close, the controller measures the book and updates its estimates. It trades only when a __trigger__ fires. A trigger starts a new allocation, a wealth forecast, and orders for the next open.

Let's build the pieces of this loop, starting with how the controller chooses target holdings.

___


## Portfolio Allocation with Trading Limits
The controller chooses target holdings in two steps. A __selection rule__ picks a basket $\mathcal N$ of $K$ stocks. A __weighting rule__ then chooses how many shares of each stock to hold.

The study ranks stocks by a score that combines each stock's expected growth with the direction of the market:

> __Definition (stock score):__ Let $\alpha_i$ and $\beta_i$ be the single index intercept (units: $\mathrm{yr}^{-1}$) and market sensitivity of stock $i$. Let $\tilde g_M$ be the 252-day exponential moving average of SPY's daily growth rate (units: $\mathrm{yr}^{-1}$). The score $\gamma_i$ and the market state $\xi$ are given by:
>
> $$
> \begin{aligned}
> \gamma_i&=\tanh\left(h\,\frac{\alpha_i+\beta_i\tilde g_M}{\max(|\beta_i|,0.1)^{\xi}}\right),\\
> \xi&=-G\left(\frac{\mathrm{EMA}_{21}}{\mathrm{EMA}_{63}}-1\right),
> \end{aligned}
> $$
> where $\mathrm{EMA}_{21}$ and $\mathrm{EMA}_{63}$ are the 21-day and 63-day exponential moving averages of SPY's close price. The scale $h=1$ year makes the argument of $\tanh$ dimensionless. The gain $G>0$ keeps $\xi$ between $-1$ and $1$ on 99% of the calibration days.

__How does the score work?__ The numerator is the stock's expected growth rate under the single index model, and the score has its sign. When SPY's 21-day average falls below its 63-day average, the market is falling and $\xi>0$. For a stock with positive expected growth, dividing by $|\beta_i|^{\xi}$ then favors a small $|\beta_i|$, because such a stock falls less with the market.

In a rising market, $\xi<0$ and the tilt favors stocks with large $|\beta_i|$. The floor of 0.1 keeps the denominator away from zero, and $\tanh$ keeps every score between $-1$ and $1$.

The study's selection by score keeps the three highest-scoring stocks in each of the 11 sectors, 33 stocks in all. We hold that basket fixed and turn to the weighting rule.

The scores become the preference weights of a Cobb–Douglas utility:

> __Proposition (Cobb–Douglas allocation):__ Consider a basket with positive scores, today's close prices $S_i$ (units: USD/share), and current wealth $W$ (units: USD). Maximizing $\sum_{i\in\mathcal N}\gamma_i\ln n_i$ over share counts $n_i>0$ and cash $c\geq0$, subject to the budget $\sum_{i\in\mathcal N}n_iS_i+c=W$, gives:
>
> $$
> \boxed{w_i^\star=\frac{\gamma_i}{\sum_{j\in\mathcal N}\gamma_j},\qquad n_i^\star=\frac{W}{S_i}\,w_i^\star,\qquad c^\star=0,}
> $$
> where $w_i^\star=n_i^\star S_i/W$ is the target fraction of wealth in stock $i$. The first-order conditions equalize marginal utility per dollar, $\gamma_i/(n_iS_i)$, across stocks.

Each stock's target weight equals its score divided by the total score. We can trace every target weight back to a stock's expected growth and market sensitivity, which is why the study calls the allocation __interpretable__.

The controller also limits concentration, volatility, and trading costs:

> __Definition (constrained utility allocation):__ Let $n_i^{\mathrm{prev}}$ be the shares the book holds before the re-plan. Choose share counts $n_i>0$ and cash $c\geq0$, with target weights $w_i=n_iS_i/W$ and a utility $U(n;\gamma)$ from the table below, to solve:
>
> $$
> \boxed{\begin{aligned}
> \operatorname*{maximize}_{n_1,\ldots,n_K>0,\,c\geq0}\quad &U(n;\gamma)\\
> \text{subject to}\quad &\sum_{i\in\mathcal N}n_iS_i+c=W &&\text{(budget)}\\
> &w_i\leq\bar w\quad\text{for each }i &&\text{(concentration)}\\
> &\sqrt{w^\top\Sigma_gw}\leq\sigma_{\mathrm{SPY}} &&\text{(volatility)}\\
> &\bar c\sum_{i\in\mathcal N}|n_i-n_i^{\mathrm{prev}}|\leq B &&\text{(trading cost)}\\
> &\sum_{i\in\mathcal N}\beta_iw_i\leq\bar\beta &&\text{(optional hedge)}.
> \end{aligned}}
> $$
>
> The concentration limit $\bar w$ is the largest fraction of wealth one stock may hold. The volatility limit keeps the book's growth rate from varying more than SPY's, where $\Sigma_g$ is the covariance of the stocks' daily growth rates (units: $\mathrm{yr}^{-2}$) and $\sigma_{\mathrm{SPY}}$ is SPY's growth-rate standard deviation over the calibration period (units: $\mathrm{yr}^{-1}$).
>
> Each share bought or sold costs $\bar c$ (units: USD/share), an estimate of half the bid–ask spread. The total trading cost may not exceed $B$ (units: USD). The optional hedge limits the book's market sensitivity to $\bar\beta=\max(0,1-\xi)$, which tightens as the market falls.

A numerical solver finds the constrained optimum. Cash can remain when a limit stops further purchases.

The study compares four utilities, whose forms and budget-only allocations are:

$$
\begin{array}{lcc}
\hline
\text{Utility} & U(n;\gamma) & \text{Budget-only allocation} \\[5pt]
\hline
\text{Cobb–Douglas} & \displaystyle\sum_i\gamma_i\ln n_i & w_i\propto\gamma_i \\[9pt]
\text{CES on shares} & \displaystyle\left(\sum_i\gamma_i n_i^{\rho}\right)^{1/\rho} & n_i\propto(\gamma_i/S_i)^{\eta} \\[9pt]
\text{CES on weights} & \displaystyle\left(\sum_i\gamma_i w_i^{\rho}\right)^{1/\rho} & w_i\propto\gamma_i^{\eta} \\[9pt]
\text{Stone–Geary} & \displaystyle\sum_i\gamma_i\ln(n_i-b_i) & (n_i-b_i)S_i\propto\gamma_i \\[5pt]
\hline
\end{array}
$$

The two constant elasticity of substitution (CES) utilities use the elasticity $\eta=0.5$, with $\rho=1-1/\eta$. The Stone–Geary utility keeps a baseline of $b_i$ shares in each stock, worth 1% of wealth.

If every score is negative and the hedge is off, Cobb–Douglas and CES hold only cash. Stone–Geary keeps its baseline shares.

___


## Keeping the Estimates Current
The score uses the single index estimates $\alpha_i$ and $\beta_i$. A fit made once would give the controller a fixed view of each stock. We use __exponentially weighted least squares__ (EWLS) to revise the fit after every close.

> __Definition (online single index estimation):__ Let $g_i(s)$ and $g_M(s)$ be the daily growth rates of stock $i$ and SPY on trading day $s$ (units: $\mathrm{yr}^{-1}$). For a trial intercept $\alpha$ and market sensitivity $\beta$, the weighted squared-error loss is:
>
> $$
> \begin{aligned}
> \mathcal L_{i,t}(\alpha,\beta)
> &=\lambda^t\mathcal L_{i,0}(\alpha,\beta)
> +\sum_{s=1}^{t}\lambda^{t-s}\left(g_i(s)-\alpha-\beta g_M(s)\right)^2,\\
> \lambda&=2^{-1/h_{\mathrm{EW}}},\qquad h_{\mathrm{EW}}=63\;\text{trading days},
> \end{aligned}
> $$
> where $\mathcal L_{i,0}$ is the loss of the initial calibration, counted as 63 days of data. Minimizing $\mathcal L_{i,t}$ gives the updated estimates $\alpha_i(t)$ and $\beta_i(t)$. The decay $\lambda$ halves an observation's weight every 63 trading days.

A shorter half-life follows changes faster but gives each observation more influence. A longer half-life averages over more history. Each update uses only prices available at that close.

The smoothed market growth $\tilde g_M$ and the market state $\xi$ also update after every close. At a re-plan, the controller rebuilds $\Sigma_g$ from the current $\beta_i$, the market's growth-rate variance, and the last 126 days of single index residuals.

___


## Model-Predictive Rebalancing
The controller decides when to trade by comparing the book with a forecast:

> __Model-predictive rebalancing:__ A controller uses a model to predict the consequences of a plan over a finite horizon. It compares new measurements with that prediction and makes a new plan from the measured state when a trigger fires.

Here, the plan is to hold the target shares for the next $T=21$ trading days. The forecast of that plan stays fixed until the next re-plan, while the estimates keep updating.

### Forecast the wealth of holding the target shares
Let $t_0$ be the re-plan day, $n_i^\star$ the target shares, and $c^\star=W(t_0)-\sum_i n_i^\star S_i(t_0)$ the cash left at that day's prices. A forecast path keeps these share counts fixed. Cash grows at the risk-free rate $g_f$ (units: $\mathrm{yr}^{-1}$) of the re-plan day.

The forecast uses a multi-asset geometric Brownian motion with time step $\Delta t=1/252$ year.

For path $p$ and forecast day $\tau=1,\ldots,T$, the controller draws the growth rates and advances each price as follows:

$$
\begin{aligned}
\mathbf g^{(p)}(\tau)&\sim\mathcal N(\boldsymbol\mu_g,\Sigma_g)
&&\text{draw the stocks' growth rates together},\\
\mu_{g,i}&=\alpha_i+\beta_i\tilde g_M
&&\text{expected growth at the re-plan},\\
S_i^{(p)}(\tau)&=S_i^{(p)}(\tau-1)\exp\left(g_i^{(p)}(\tau)\Delta t\right)
&&\text{compound from }S_i^{(p)}(0)=S_i(t_0).
\end{aligned}
$$

The mean $\boldsymbol\mu_g$ and covariance $\Sigma_g$ stay at their re-plan values. Each day and each path uses an independent draw.

The wealth on path $p$ is given by:

$$
W^{(p)}(\tau)=\sum_i n_i^\star S_i^{(p)}(\tau)+c^\star\exp(g_f\tau\Delta t).
$$

We draw $N_p=1{,}000$ paths. On each forecast day $\tau$, the paths give a sample mean $\mu_W(\tau)$ and standard deviation $\sigma_W(\tau)$ of wealth (units: USD).

Under a normal approximation, the 95% __forecast band__ is given by:

$$
\boxed{\mu_W(\tau)-1.96\sigma_W(\tau)\;\leq\;W(t_0+\tau)\;\leq\;\mu_W(\tau)+1.96\sigma_W(\tau).}
$$

### Three reasons to make a new plan
After each close, the controller updates the peak wealth $W^{\mathrm{pk}}$ and counts the trading days $\tau$ since the last re-plan. It then checks three triggers in order:

1. __Drawdown:__ Re-plan if wealth has fallen more than $D_{\max}=8\%$ below its peak, $1-W(t)/W^{\mathrm{pk}}>D_{\max}$.
2. __Horizon:__ Re-plan if $\tau\geq T$, because the forecast has reached its end.
3. __Band exit:__ Re-plan if measured wealth lies outside the forecast band for day $\tau$.

After a drawdown re-plan, the peak resets to current wealth. The next drawdown trigger then responds only to a new loss.

__Why check both sides of the band?__ An unusually large gain also means the forecast no longer describes the book. The re-plan recomputes the best holdings from current estimates. The trigger itself does not say whether to buy or sell.

Let's look at three plans of one real book:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13b-ForecastBand.svg" width="860" style="max-width:100%; height:auto;" alt="The wealth of one book divided by its starting wealth, in red, over 50 trading days from October 10 to December 22, 2025, rising from about 1.02 to 1.15. Three blue regions are the 95 percent forecast bands of three plans, each with a dashed forecast mean, starting October 10, November 10, and November 20. Gold rings mark the re-plans. Those on November 10 and December 22 are labeled 21 days. The one on November 20 is labeled left the band, and a note says the wealth was 2.1 standard deviations below the mean, just under the lower edge of the second band."/>
    </center>
</div>

__Three plans on real prices.__ The book holds 13 stocks picked by hand without 2025 data, with Cobb–Douglas utility and the hedge off. The red line is wealth divided by starting wealth. Each blue region is the 95% band of one plan around its dashed forecast mean.

The first and third plans ended at the 21-day horizon. The second ended on November 20, when wealth closed 2.1 standard deviations below its forecast mean, just outside the band. Each new band starts from the wealth measured at that re-plan.

### The daily controller algorithm

__Initialize__: Given the daily prices and risk-free growth rates, a fixed basket $\mathcal N$, initial model estimates, starting wealth $W_0$, horizon $T$, drawdown threshold $D_{\max}$, forecast path count $N_p$, and final trading day $t_{\mathrm{end}}$.

Set $n_i\gets0$ shares for every stock $i\in\mathcal N$, cash $c\gets W_0$, the tracked peak $W^{\mathrm{pk}}\gets W_0$, and the forecast age $\tau\gets0$. Start with no forecast and no pending orders.

For each trading day $t=1,2,\dots,t_{\mathrm{end}}$ __do__:

1. __Open:__ If orders are pending, __then__ fill them at the open, selling first and scaling purchases to available cash after costs. Update the shares $n_i$ and cash $c$, then clear the pending orders.
2. __Advance:__ If $t>1$, __then__ accrue cash $c\gets c\exp\left(g_f(t)\Delta t\right)$ and advance the forecast age $\tau\gets\tau+1$.
3. __Measure:__ At the close, compute the wealth $W(t)\gets\sum_{i\in\mathcal N}n_iS_i(t)+c$ and update the tracked peak $W^{\mathrm{pk}}\gets\max\left(W^{\mathrm{pk}},W(t)\right)$.
4. __Estimate:__ Update the single index estimates, $\tilde g_M$, and $\xi$ using observations through this close. Compute the stock scores $\gamma_i$.
5. __Trigger:__ If $t=1$, __then__ re-plan, with the initial allocation as the reason. Otherwise, take the first trigger that holds:
    - __Drawdown:__ If $1-W(t)/W^{\mathrm{pk}}>D_{\max}$, __then__ re-plan.
    - __Horizon:__ If $\tau\geq T$, __then__ re-plan.
    - __Band exit:__ If a forecast band exists and $W(t)$ lies outside it for day $\tau$, __then__ re-plan.

    Record the trigger as the re-plan reason. If none holds, __then__ record no re-plan, keep the shares and forecast, and go to step 9.
6. __Allocate:__ Solve the constrained allocation from the current account for target shares $n_i^\star$ and cash $c^\star$. If the solve fails, __then__ set $n_i^\star\gets n_i$ and $c^\star\gets c$ and place no orders. Otherwise, if $t<t_{\mathrm{end}}$, __then__ queue the orders $\Delta n_i\gets n_i^\star-n_i$ for the next open.
7. __Forecast:__ Simulate $N_p$ wealth paths of the target holdings over $T$ days. If the target book holds stock, __then__ store the forecast band. Otherwise, clear the band. An all-cash book has no band to exit, and the horizon trigger revisits it after $T$ days.
8. __Reset:__ Set $\tau\gets0$. If the re-plan reason is drawdown, __then__ reset the tracked peak $W^{\mathrm{pk}}\gets W(t)$.
9. __Record:__ Store the day's wealth, holdings, trades, forecast, and re-plan reason.

__Output__: The daily account history and the re-plan log.

Let's run the controller on real prices:

> __Example:__
>
> [▶ Trace an adaptive portfolio on real data](CHEME-5660-L13b-Example-MPC-RealData-Fall-2026.ipynb). In this example, we compute the market state and stock scores, run the constrained allocator inside a daily controller, and read its wealth and decision log. We then compare weighting rules, stock selection, and the hedge on the same historical path.

Next, we test which of the controller's choices mattered.

___

## Testing the Controller
Does the controller meet our goal of making money when it can and preserving cash when it cannot? Each test changes one choice and keeps the market path, costs, and starting wealth fixed.

The study scores each run by its terminal wealth ratio. The examples also report each run's maximum drawdown:

$$
\begin{aligned}
R&=\frac{W(t_{\mathrm{end}})}{W_0},\\
\operatorname{MDD}&=\max_t\left(1-\frac{W(t)}{\max_{u\leq t}W(u)}\right),
\end{aligned}
$$
where $W_0$ is the starting wealth, $t_{\mathrm{end}}$ the final trading day, and $u$ runs over trading days up to $t$. Unlike the drawdown trigger, this peak never resets.

### The test markets
The real path trades the 373 stocks from January 2025 to September 2026, with the inputs calibrated on 2014 to 2024.

The synthetic markets come from the multi-asset JumpHMM generator, fitted to the same 373 stocks. Each synthetic market calibrates on its own 2014 to 2024 and trades its own 2025 and 2026.

The study drew 10,000 synthetic markets and built three sets:

* __Typical:__ This set holds 600 markets drawn at random.
* __Burst sell-offs:__ This set holds the 200 deepest SPY drawdowns among markets with a jump burst, months of large moves of both signs.
* __Bear sell-offs:__ This set holds the 200 deepest SPY drawdowns among markets without a burst, where prices fall more slowly.

The sell-off sets show how the controller behaves __given__ a severe decline. Their results do not estimate how often such declines occur.

### What the tests show
Each entry gives the range of median $R$ over seven weighting rules, including the four utilities, with the hedge off:

| Basket | Real path | Typical | Burst sell-offs | Bear sell-offs |
|:---|:---:|:---:|:---:|:---:|
| 33 stocks, the 3 highest scores per sector | 1.09–1.23 | 1.14–1.23 | 0.97–1.00 | 0.81–0.87 |
| 13 highest-scoring stocks | 1.25–1.52 | 1.20–1.24 | 0.98–1.01 | 0.84–0.86 |
| 13 random stocks | 1.03–1.16 | 1.10–1.17 | 0.93–0.99 | 0.77–0.98 |
| 13 stocks picked by hand without 2025 data | 1.44–1.70 | 1.18–1.24 | 0.95–1.03 | 0.79–1.00 |
| SPY bought and held | 1.32 | 1.19 | 0.86 | 0.72 |
| Equal-weight portfolio of all 373 stocks | 1.30 | 1.23 | 0.92 | 0.75 |

__What does the real path show?__ The basket moved terminal wealth more than the weighting rule did. The 33 score-picked stocks ended below SPY and the equal-weight portfolio. The hand-picked 13 ended highest.

__Was the hand-picked result luck?__ In typical markets, the hand-picked 13 ended inside the range of the score-picked baskets. The real period was also good for the market: real SPY ended above 70% of the typical synthetic SPY paths. One path cannot tell a good rule from a lucky one.

__Does selection by score help?__ In typical markets, the 13 highest-scoring stocks beat 13 random stocks in 62 to 71% of the markets. In bear sell-offs, they beat random picks in only 28 to 38% of the markets for five of the seven weighting rules. The generator keeps each stock's past trend, which favors selection by score more than real prices did.

__Does the controller preserve cash in a sell-off?__ In both sell-off sets, every basket ended above SPY and the equal-weight portfolio, random baskets included. Part of the margin came from the simulated market, because the equal-weight portfolio also beat SPY. With the hedge off, every book also held an average beta below one and fell less than the market.

__Does the hedge pay?__ The hedge lowered terminal wealth on real prices and in typical markets. In the bear set, the hedge raised terminal wealth in 65 of 70 combinations of basket and weighting rule.

Let's run the controller on synthetic markets:

> __Example:__
>
> [▶ Test the controller across synthetic markets](CHEME-5660-L13b-Example-MPC-SyntheticData-Fall-2026.ipynb). In this example, we draw typical markets and two kinds of sell-off from a fitted generator. We run the same controller on each market and compare its outcomes with SPY and an equal-weight portfolio.

The example uses far fewer markets than the study, which makes its counts rougher.

___


## Summary
We built a trading system that updates its estimates every day and trades only when a trigger fires. We then tested which of its choices mattered on real and synthetic markets.

> __Key Takeaways:__
>
> * __Adaptation begins with the scores:__ We updated the single index fit with exponentially weighted least squares after every close. The scores combined those estimates with the market state, while the budget and trading limits set the target holdings.
> * __Feedback determines when a target becomes a trade:__ We compared measured wealth with the forecast of holding the current plan. A drawdown, the horizon, or a band exit started a new allocation, with orders filled at the next open.
> * __Matched tests identify what each choice adds:__ On one real path, the basket mattered more than the weighting rule. In synthetic sell-offs, every book's median ended above SPY, random baskets included.

The real-data example applies the controller to one observed market. The synthetic example repeats it across markets to examine how its outcomes depend on the conditions it meets.

___


## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success.

Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___
